In [26]:
import numpy as np

# Create a random number generator with a fixed seed so results are reproducible
rng = np.random.default_rng(1)

# n = width of the neural network layer (number of neurons)
# d = input dimension
# Both are large to approximate the "infinite width" assumption in tensor programs
n = 20000
d = 20000

# Variance parameter for weight initialization
sigma2_W_0 = 1.0

# Generate a dataset of n input vectors, each of dimension d
# Each element is drawn from a standard normal distribution N(0,1)
X = rng.standard_normal((n, d))

# Initialize the first layer weight matrix W_0
# Shape: (d, n) so inputs (size d) map to hidden layer (size n)
# Scaling by sqrt(1/d) keeps variance stable as dimension grows
W_0 = rng.standard_normal((d, n)) * np.sqrt(sigma2_W_0 / d)

# Initialize second layer weight vector "a"
# Shape: (1, n) so it collapses hidden layer to a scalar
# Scaling by sqrt(1/n) ensures variance stays controlled
a = rng.standard_normal((1, n)) * np.sqrt(sigma2_W_0 / n)
print(f"W_0 shape: {W_0.shape}, a shape: {a.shape}")

# Select two input samples from the dataset
# We track two inputs to study correlations between them
x_1 = X[0]
x_2 = X[1]
print(f"x_1 shape: {x_1.shape}, x_2 shape: {x_2.shape}")


# FIRST LAYER PREACTIVATIONS 

# Compute preactivations for the first input
# Equivalent to a fully connected layer: g1 = x * W
g1_1 = x_1 @ W_0

# Compute preactivations for the second input
g1_2 = x_2 @ W_0


#  RELU ACTIVATION 

# Apply ReLU activation to the preactivations
# ReLU(x) = max(0, x)
h1_1 = np.maximum(0, g1_1)
h1_2 = np.maximum(0, g1_2)


#  SECOND LAYER 

# Compute scalar output of second layer for input 1
# a @ h1 collapses the hidden layer into a single value
g2_1 = (a @ h1_1).item()

# Same computation for input 2
g2_2 = (a @ h1_2).item()


#  ADDITIONAL RANDOM NOISE 

# Sample two independent Gaussian values
# These act as noise terms in the tensor program
y_1, y_2 = rng.standard_normal(2)

# Combine noise and scaled network output
# This defines the next tensor variable in the tensor program
h2_1 = y_1 - (1 / np.sqrt(d)) * g2_1
h2_2 = y_2 - (1 / np.sqrt(d)) * g2_2
print(f"h2_1: {h2_1:.6f}, h2_2: {h2_2:.6f}")

g3_1 = (a.T * h2_1)
g3_2 = (a.T * h2_2)
print(f"g3_1 shape: {g3_1.shape}, g3_2 shape: {g3_2.shape}")


#  RELU DERIVATIVE INDICATOR 

# Compute derivative of ReLU
# ReLU'(x) = 1 if x > 0 else 0
h3_1 = (g1_1 > 0).astype(np.float64)
h3_2 = (g1_2 > 0).astype(np.float64)
print(f"h3_1 shape: {h3_1.shape}, h3_2 shape: {h3_2.shape}")

#  COMBINED TENSOR VARIABLE 

# Compute a scalar projection of the ReLU derivative pattern
# Then multiply elementwise with the same pattern
# This creates another tensor variable used in the TP analysis
h4_1 = (1 / np.sqrt(d)) * (g3_1 * h3_1)
h4_2 = (1 / np.sqrt(d)) * (g3_2 * h3_2)
print(f"h4_1 shape: {h4_1.shape}, h4_2 shape: {h4_2.shape}")


#  MORE SCALAR STATISTICS 

# Compute interaction between input vector and h4
g4_1 = (1 / n) * ([x_1] @ h4_1.T)
g4_2 = (1 / n) * ([x_2] @ h4_2.T)
print(f"g4_1 shape: {g4_1.shape}, g4_2 shape: {g4_2.shape}")

g3_1 = (a.T * h2_1).flatten()
g3_2 = (a.T * h2_2).flatten()


#  COLLECT ALL TENSOR VARIABLES 

# Each element in this list is treated as a random vector of length n
# Tensor programs predict their joint Gaussian distribution
h_vectors = [
    g1_1,
    g1_2,
    np.full(n, g2_1),
    np.full(n, g2_2),
    np.full(n, g3_1),
    np.full(n, g3_2),
    np.full(n, g4_1),
    np.full(n, g4_2)
]


#  THEORETICAL COVARIANCE MATRICES 

# Covariance of first layer activations
B1 = np.eye(2)

# Covariance of ReLU outputs
# Known analytic formula when inputs are Gaussian
B2 = np.array([
    [1/2, 1/(2*np.pi)],
    [1/(2*np.pi), 1/2]
])

# Covariance for later tensor variables
B3 = np.array([
    [1 + 1/d, 1/(2*d*np.pi)],
    [1/(2*d*np.pi), 1 + 1/d]
])

# Diagonal entry for final block
b4_diag = 1/2 + 5/(8*d) - 1/(4*d*np.pi)

# Final covariance block
B4 = np.array([
    [b4_diag, 1/(2*d*np.pi)],
    [1/(2*d*np.pi), b4_diag]
])


#  FULL GAUSSIAN COVARIANCE MATRIX 

# Assemble full covariance matrix for all 8 variables
Sigma = np.block([
    [B1, np.zeros((2,6))],
    [np.zeros((2,2)), B2, np.zeros((2,4))],
    [np.zeros((2,4)), B3, np.zeros((2,2))],
    [np.zeros((2,6)), B4]
])


#  SAMPLE FROM THEORETICAL GAUSSIAN 

def sample_Zs(n_samples, rng=None):
    """
    Generate samples from the predicted multivariate Gaussian
    with covariance Sigma.
    """
    rng = rng or np.random.default_rng()
    return rng.multivariate_normal(np.zeros(8), Sigma, size=n_samples)


#  MASTER THEOREM VERIFICATION 

def master_theorem_check(h_vectors, sample_Zs, psi_list, n_Z_samples=50_000):
    """
    Check Yang's Master Theorem by comparing:

    LHS = empirical average across neurons
    RHS = expectation under predicted Gaussian
    """

    # Sample Gaussian vectors from the theoretical distribution
    Z_all = sample_Zs(n_Z_samples)

    # Number of tensor variables
    k = len(h_vectors) if h_vectors is not None else Z_all.shape[1]

    Z_samples = Z_all[:, :k]

    results = []

    for name, psi in psi_list:

        # RHS: Monte Carlo estimate of expectation under Gaussian
        rhs = np.mean([psi(*Z_samples[i]) for i in range(n_Z_samples)])

        if h_vectors is not None and len(h_vectors) == k:

            n = len(h_vectors[0])

            # LHS: empirical average across neuron coordinates
            lhs = (1 / n) * sum(
                psi(*(h_vectors[j][α] for j in range(k)))
                for α in range(n)
            )

            results.append((name, lhs, rhs, np.isclose(lhs, rhs, atol=0.05)))

        else:
            results.append((name, None, rhs, None))

    return results


#  TEST FUNCTIONS 

psi_list = [

    # Sum of all variables
    ("E[sum Zg_i]", lambda *zs: sum(zs)),

    # Sum of squares
    ("E[sum Zg_i^2]", lambda *zs: sum(z**2 for z in zs)),

    # Specific cross interactions
    ("E[Zg1_1*Zg2_1+...+Zg4_2^2]",
     lambda *zs: zs[0]*zs[2] + zs[1]*zs[3] + zs[4]*zs[5] + zs[6]*zs[7]),

    # Square of the sum
    ("E[(sum Zg_i)^2]", lambda *zs: sum(zs)**2),

    # Pairwise interaction test
    ("E[sum of pairs Zg_i*Zg_j]",
     lambda *zs: zs[0]*zs[6] + zs[1]*zs[7] + zs[2]*zs[4] + zs[3]*zs[5]),

    # Product test
    ("E[prod (1+Zg_i)]",
     lambda *zs: np.prod([1 + z for z in zs])),
]


#  RUN THE EXPERIMENT 

out = master_theorem_check(h_vectors, sample_Zs, psi_list)

# Print comparison between empirical and theoretical expectations
for name, lhs, rhs, ok in out:
    print(f"{name}:  LHS={lhs:.6f}  RHS={rhs:.6f}  ok={ok}")

W_0 shape: (20000, 20000), a shape: (1, 20000)
x_1 shape: (20000,), x_2 shape: (20000,)
h2_1: -0.863573, h2_2: 1.099711
g3_1 shape: (20000, 1), g3_2 shape: (20000, 1)
h3_1 shape: (20000,), h3_2 shape: (20000,)
h4_1 shape: (20000, 20000), h4_2 shape: (20000, 20000)
g4_1 shape: (1, 20000), g4_2 shape: (1, 20000)
E[sum Zg_i]:  LHS=-2.026640  RHS=-0.016516  ok=False
E[sum Zg_i^2]:  LHS=4.082293  RHS=6.016766  ok=False
E[Zg1_1*Zg2_1+...+Zg4_2^2]:  LHS=0.005219  RHS=-0.004117  ok=True
E[(sum Zg_i)^2]:  LHS=6.094911  RHS=6.290907  ok=False
E[sum of pairs Zg_i*Zg_j]:  LHS=-0.000003  RHS=-0.004211  ok=True
E[prod (1+Zg_i)]:  LHS=-0.032316  RHS=1.046350  ok=False
